In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
from selenium.common.exceptions import TimeoutException

import time
import pandas as pd
import sqlite3

In [4]:
service = Service()
options = Options()
options.headless = True
driver = webdriver.Firefox(service=service, options=options)

url = "https://nootropicsdepot.com/all-products/?_bc_fsnf=1&Form=Powders&sort=alphaasc"

all_products = []

while url:
    driver.get(url)
    time.sleep(2)  # Wait for page to load

    products = driver.find_elements(By.CSS_SELECTOR, ".productList .listItem")
    for product in products:
        try:
            title_elem = product.find_element(By.CSS_SELECTOR, ".listItem-title a")
            title = title_elem.text.strip()
            href = title_elem.get_attribute("href")
            all_products.append({"title": title, "href": href})
        except NoSuchElementException:
            continue

    # Find next page link
    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, ".pagination-item--next a.pagination-link")
        url = next_btn.get_attribute("href")
    except NoSuchElementException:
        url = None  # No more pages

# Convert to DataFrame and display/save
df = pd.DataFrame(all_products)
# print(df)

In [5]:
# For each product, visit its page and extract size options and prices

# Re-initialize the driver since it was quit in a previous cell
# service = Service()
# options.headless = True
# driver = webdriver.Firefox(service=service, options=options)

product_options = []

for idx, row in df.iterrows():
    driver.get(row['href'])
    time.sleep(2)  # Wait for product page to load

    options = []
    try:
        # Find all size option wrappers
        option_wrappers = driver.find_elements(By.CSS_SELECTOR, ".form-field__size .form-option-wrapper")
        for wrapper in option_wrappers:
            try:
                # Get size label
                size = wrapper.find_element(By.CSS_SELECTOR, ".form-option-variant").text.strip()
                # Select the radio button to update the price
                radio = wrapper.find_element(By.CSS_SELECTOR, "input[type='radio']")
                driver.execute_script("arguments[0].click();", radio)
                time.sleep(1)  # Wait for price to update

                # Get price
                price_elem = driver.find_element(By.CSS_SELECTOR, "span[data-product-price-without-tax].price--withoutTax")
                price = price_elem.text.strip()
                options.append({"size": size, "price": price})
            except NoSuchElementException:
                continue
    except NoSuchElementException:
        options = []

    product_options.append(options)

# Add the options/prices as a new column in the DataFrame
df["options"] = product_options

print(df)
# Optionally save to CSV
# df.to_csv("nootropicsdepot_products_with_options.csv", index=False)
driver.quit()

                                            title  \
0                     (-)- Epicatechin 90% Powder   
1               7,8 DHF Powder | Dihydroxyflavone   
2    AAKG 2:1 Powder | L-Arginine A-Ketoglutarate   
3                 Acetyl L-Carnitine ALCAR Powder   
4                         Agmatine Sulfate Powder   
..                                            ...   
126          Tongkat Ali Powder | 2% Eurycomanone   
127                 Triacetyluridine Powder | TAU   
128           Turkey Tail Mushroom Extract Powder   
129  Uridine Monophosphate Powder | Disodium Salt   
130                    White Jelly Extract Powder   

                                                  href  \
0      https://nootropicsdepot.com/epicatechin-powder/   
1    https://nootropicsdepot.com/7-8-dihydroxyflavo...   
2    https://nootropicsdepot.com/aakg-2-1-powder-l-...   
3    https://nootropicsdepot.com/acetyl-l-carnitine...   
4    https://nootropicsdepot.com/agmatine-sulfate-p...   
..             

In [7]:

import json

# Convert the 'options' column to JSON strings
df["options"] = df["options"].apply(json.dumps)

# Save DataFrame to SQLite database
conn = sqlite3.connect("nootropicsdepot_products.db")
df.to_sql("products", conn, if_exists="replace", index=False)
conn.close()

In [8]:
import sqlite3
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.common.exceptions import NoSuchElementException
import time

# Load product URLs from the database
conn = sqlite3.connect("nootropicsdepot_products.db")
df = pd.read_sql("SELECT * FROM products", conn)
conn.close()

service = Service()
options = Options()
options.headless = True
driver = webdriver.Firefox(service=service, options=options)

dosage_info = []

for idx, row in df.iterrows():
    url = row['href']
    driver.get(url)
    time.sleep(2)  # Wait for page to load

    # Check if the Product Overview tab is active
    try:
        active_tab = driver.find_element(By.CSS_SELECTOR, "ul.tabs li.tab.is-active a.tab-title[href='#tab-description']")
    except NoSuchElementException:
        dosage_info.append("")
        continue

    # Find all headings in the description tab
    try:
        headings = driver.find_elements(By.CSS_SELECTOR, "#tab-description h2.Heading")
        found = False
        for heading in headings:
            if "dosage" in heading.text.lower():
                # Get the next sibling <p>
                p_elem = heading.find_element(By.XPATH, "following-sibling::p[1]")
                dosage_info.append(f"{heading.text.strip()} {p_elem.text.strip()}")
                found = True
                break
        if not found:
            dosage_info.append("")
    except NoSuchElementException:
        dosage_info.append("")

driver.quit()

# Add dosage info to DataFrame and save
df["dosage"] = dosage_info

conn = sqlite3.connect("nootropicsdepot_products.db")
df.to_sql("products", conn, if_exists="replace", index=False)
conn.close()